# TabResNet DLBAC — HyConEx / HyperLogic datasets

Deuxième notebook : **dernière version du modèle** (`TabResNetDLBACTrainer`).

| Dimension entrée | Stratégie |
|---|---|
| ≤ 512 features | **instance** — TabResNet par échantillon + DR-Net + CF binaire (Dry Bean) |
| > 512 features | **bipolar_hyper** — encodeur + hyper TabResNet 78% + règles DR-Net 22% + CF continu |

**Jeux** : `HyConEx/data/` + `HyperLogic/data/` (même chargeurs que le notebook DR-HyperCF).

**Prétraitement** : continus → intervalles ; catégorielles → one-hot ; puis bipolarisation.

Pour chaque jeu : métriques, mode utilisé, contrefactuels et règles.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from hyconex_hyperlogic_datasets import list_available_datasets
from hyconex_pure_bipolar import explain_counterfactual_bipolar
from hyconex_pure_bipolar.bipolar import continuous_to_bipolar
from tabresnet_dlbac import TabResNetDLBACConfig, TabResNetDLBACTrainer
from tabular_mixed_preprocessing import TabularDatasetSplits, get_dataset_loaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Jeux détectés:', list_available_datasets())

Device: cuda
Jeux détectés: {'hyconex': ['adult', 'audit', 'blobs', 'compas', 'german_credit', 'heloc', 'law', 'moons', 'wine', 'moons_with_blob'], 'hyperlogic': ['cardio', 'disease', 'brca-n', 'brca-s']}


## Configuration

In [2]:
ALL_LOADERS = get_dataset_loaders()

DATASETS = None  # None = tous ; ex. ['hyconex/adult', 'hyperlogic/cardio']
if DATASETS is None:
    DATASETS = sorted(ALL_LOADERS.keys())

SEED = 42
QUICK_RUN = False
MAX_INSTANCE_DIM = 512

print(f'{len(DATASETS)} jeux:', DATASETS)

14 jeux: ['hyconex/adult', 'hyconex/audit', 'hyconex/blobs', 'hyconex/compas', 'hyconex/german_credit', 'hyconex/heloc', 'hyconex/law', 'hyconex/moons', 'hyconex/moons_with_blob', 'hyconex/wine', 'hyperlogic/brca-n', 'hyperlogic/brca-s', 'hyperlogic/cardio', 'hyperlogic/disease']


In [3]:
def make_tabresnet_config(splits: TabularDatasetSplits) -> TabResNetDLBACConfig:
    n_feat = splits.X_train.shape[1]
    n_classes = len(splits.class_names)
    high = n_feat > MAX_INSTANCE_DIM

    inst_epochs = 8 if QUICK_RUN else (45 if n_feat < 100 else 35)
    bipolar_epochs = 6 if QUICK_RUN else (50 if high else 40)

    return TabResNetDLBACConfig(
        seed=SEED,
        max_instance_dim=MAX_INSTANCE_DIM,
        instance_epochs=inst_epochs,
        instance_batch_size=128,
        instance_num_rules=min(128, 32 + 6 * n_classes),
        instance_temperature=0.7,
        instance_cf_lambda=0.10 if n_classes > 2 else 0.15,
        instance_flip_lambda=0.05,
        instance_cf_warmup=2 if not QUICK_RUN else 0,
        embed_epochs=bipolar_epochs,
        embed_batch_size=96 if high else 64,
        embed_num_rules=48,
        hyper_hidden_dim=128,
        input_encoding=splits.input_encoding,
        early_stop_metric='auto',
    )


def train_on_dataset(key: str) -> dict:
    splits = ALL_LOADERS[key]()
    cfg = make_tabresnet_config(splits)
    trainer = TabResNetDLBACTrainer(cfg, device=device)

    print(f"\n{'=' * 72}\n{key} | train {splits.X_train.shape} | classes {len(splits.class_names)}\n{'=' * 72}")

    result = trainer.fit(
        splits.X_train,
        splits.y_train,
        x_val=splits.X_val,
        y_val=splits.y_val,
        feature_names=splits.feature_names,
        class_names=splits.class_names,
        verbose=True,
    )
    metrics = trainer.evaluate(splits.X_test, splits.y_test, counterfactuals=True)
    rules = trainer.export_rules(top_per_rule=4, min_abs_weight=0.05)

    return {
        'key': key,
        'splits': splits,
        'trainer': trainer,
        'result': result,
        'metrics': metrics,
        'rules': rules,
    }


def metrics_row(run: dict) -> dict:
    m = run['metrics']
    cf = m.get('counterfactuals', {})
    return {
        'dataset': run['key'],
        'mode': run['result'].mode,
        'best_val_acc': run['result'].best_val_accuracy,
        'best_val_auroc': run['result'].best_val_auroc,
        'test_acc': m.get('accuracy'),
        'test_auroc': m.get('auroc_ovr'),
        'cf_validity': cf.get('validity_cf'),
        'cf_changed_bits': cf.get('changed_bits_mean', cf.get('hamming_mean')),
        'cf_l1': cf.get('proximity_l1_cont_mean', cf.get('proximity_l1_mean')),
        'n_rules': len(run['rules']),
    }

## Entraînement

In [4]:
runs: dict[str, dict] = {}
failures: dict[str, str] = {}

for ds_key in DATASETS:
    try:
        runs[ds_key] = train_on_dataset(ds_key)
    except Exception as exc:
        failures[ds_key] = str(exc)
        print('ERREUR', ds_key, '→', exc)

if runs:
    summary_df = pd.DataFrame([metrics_row(r) for r in runs.values()])
    display(summary_df.round(4))
else:
    print('Aucun jeu entraîné.')

if failures:
    print('\nÉchecs:')
    for k, v in failures.items():
        print('-', k, ':', v)


hyconex/adult | train (20838, 34) | classes 2
  Strategie: TabResNet instance (HybridDRNetModel)
[Epoch 001/45] [warmup] loss=0.4794 val_acc=0.7658 val_auroc=0.8791 best(accuracy)=0.7658
[Epoch 002/45] [warmup] loss=0.4413 val_acc=0.7397 val_auroc=0.8793 best(accuracy)=0.7658
[Epoch 003/45] loss=0.5794 val_acc=0.8042 val_auroc=0.8780 best(accuracy)=0.8042
[Epoch 004/45] loss=0.5192 val_acc=0.7620 val_auroc=0.8795 best(accuracy)=0.8042
[Epoch 005/45] loss=0.5079 val_acc=0.8259 val_auroc=0.8812 best(accuracy)=0.8259
[Epoch 006/45] loss=0.5013 val_acc=0.7572 val_auroc=0.8798 best(accuracy)=0.8259
[Epoch 007/45] loss=0.4963 val_acc=0.7808 val_auroc=0.8802 best(accuracy)=0.8259
[Epoch 008/45] loss=0.4965 val_acc=0.7764 val_auroc=0.8793 best(accuracy)=0.8259
[Epoch 009/45] loss=0.4952 val_acc=0.7553 val_auroc=0.8781 best(accuracy)=0.8259
[Epoch 010/45] loss=0.4970 val_acc=0.7764 val_auroc=0.8799 best(accuracy)=0.8259
[Epoch 011/45] loss=0.4873 val_acc=0.7672 val_auroc=0.8811 best(accuracy)=

D:\ecole\master 2\recherche\INN\projet\HyConEx\counterfactuals\datasets\compas.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  raw_data["length_of_stay"].fillna(
D:\ecole\master 2\recherche\INN\projet\HyConEx\counterfactuals\datasets\compas.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

[Epoch 001/45] [warmup] loss=1.1242 val_acc=0.5297 val_auroc=0.7490 best(accuracy)=0.5297
[Epoch 002/45] [warmup] loss=0.9160 val_acc=0.5356 val_auroc=0.7529 best(accuracy)=0.5356
[Epoch 003/45] loss=1.0960 val_acc=0.5415 val_auroc=0.7461 best(accuracy)=0.5415
[Epoch 004/45] loss=1.0569 val_acc=0.5401 val_auroc=0.7515 best(accuracy)=0.5415
[Epoch 005/45] loss=1.0407 val_acc=0.5341 val_auroc=0.7537 best(accuracy)=0.5415
[Epoch 006/45] loss=1.0261 val_acc=0.5490 val_auroc=0.7538 best(accuracy)=0.5490
[Epoch 007/45] loss=1.0204 val_acc=0.5534 val_auroc=0.7440 best(accuracy)=0.5534
[Epoch 008/45] loss=1.0110 val_acc=0.5371 val_auroc=0.7419 best(accuracy)=0.5534
[Epoch 009/45] loss=1.0058 val_acc=0.5460 val_auroc=0.7492 best(accuracy)=0.5534
[Epoch 010/45] loss=0.9870 val_acc=0.5341 val_auroc=0.7454 best(accuracy)=0.5534
[Epoch 011/45] loss=0.9942 val_acc=0.5475 val_auroc=0.7402 best(accuracy)=0.5534
[Epoch 012/45] loss=0.9844 val_acc=0.5638 val_auroc=0.7453 best(accuracy)=0.5638
[Epoch 013

,dataset,mode,best_val_acc,best_val_auroc,test_acc,test_auroc,cf_validity,cf_changed_bits,cf_l1,n_rules
0,hyconex/adult,instance,0.8259,0.8812,0.8231,0.8846,0.7593,15.5417,15.5417,44
1,hyconex/audit,instance,1.0000,1.0000,1.0000,1.0000,0.0000,24.3689,2.4675,44
2,hyconex/blobs,instance,0.9708,0.9978,0.9433,0.9974,0.6267,3.5867,0.7693,50
3,hyconex/compas,instance,0.5682,0.7396,0.5368,0.7102,0.4703,13.6859,13.6859,50
4,hyconex/german_credit,instance,0.6562,0.6801,0.6917,0.7689,0.3083,38.4000,37.6667,44
5,hyconex/heloc,instance,0.7212,0.7822,0.7182,0.7833,0.2818,37.9356,5.0773,44
6,hyconex/law,instance,0.7219,0.7855,0.7590,0.8557,0.2500,6.4437,0.7899,44
7,hyconex/moons,instance,0.9634,0.9874,0.9805,0.9850,0.9805,4.5756,0.4684,44
8,hyconex/moons_with_blob,instance,0.8938,0.9398,0.8800,0.9360,0.3700,4.0550,0.4142,44
9,hyperlogic/brca-n,instance,0.5000,0.5000,0.4889,0.5000,0.5111,64.9556,0.0000,44



Échecs:
- hyconex/wine : list index out of range


## Helpers contrefactuels et règles

In [ ]:
def build_cf_table_instance(run: dict, n_examples: int = 3, seed: int = 42) -> pd.DataFrame:
    trainer = run['trainer']
    splits = run['splits']
    model = trainer.model
    assert model is not None

    x_test_bin = trainer.binarizer.transform(splits.X_test)
    x_test_t = torch.tensor(x_test_bin, dtype=torch.float32, device=trainer.device)
    with torch.no_grad():
        y_pred = torch.argmax(model.predict_logits(x_test_t), dim=1).cpu().numpy()

    pool = np.where(y_pred == splits.y_test)[0]
    rng = np.random.default_rng(seed)
    n = min(n_examples, len(pool))
    idx = rng.choice(pool, size=n, replace=False) if n > 0 else np.array([], dtype=int)
    if idx.size == 0:
        return pd.DataFrame()

    x_sub = torch.tensor(x_test_bin[idx], dtype=torch.float32, device=trainer.device)
    y_pred_orig = torch.tensor(y_pred[idx], dtype=torch.long, device=trainer.device)
    bin_names = trainer.binarizer.binary_feature_names()
    class_names = splits.class_names

    with torch.no_grad():
        x_cf_all, logits_cf_all = model.generate_counterfactuals_all_classes(x_sub, y_pred_orig)
        y_pred_cf = torch.argmax(logits_cf_all, dim=2).cpu().numpy()

    x_sub_np = x_sub.cpu().numpy()
    x_cf_np = x_cf_all.cpu().numpy()
    rows = []
    for i, sample_idx in enumerate(idx):
        src = int(y_pred_orig[i].item())
        for target in range(len(class_names)):
            if target == src:
                continue
            x_cf = x_cf_np[i, target]
            diff = np.where(np.abs(x_sub_np[i] - x_cf) > 1e-6)[0]
            flips = [f"{bin_names[j]}: {int(x_sub_np[i,j])}→{int(x_cf[j])}" for j in diff[:6]]
            if len(diff) > 6:
                flips.append(f"…+{len(diff)-6}")
            rows.append({
                'sample': int(sample_idx),
                'pred': class_names[src],
                'target': class_names[target],
                'valid_cf': int(y_pred_cf[i, target] == target),
                'n_flips': len(diff),
                'flips': '; '.join(flips),
            })
    return pd.DataFrame(rows)


def build_cf_table_bipolar(run: dict, n_examples: int = 3, seed: int = 42) -> pd.DataFrame:
    trainer = run['trainer']
    splits = run['splits']
    assert trainer._bipolar is not None

    x_test = np.asarray(splits.X_test, dtype=np.float32)
    x_bin = continuous_to_bipolar(x_test)
    with torch.no_grad():
        x_t = torch.tensor(x_bin, dtype=torch.float32, device=trainer.device)
        y_pred = trainer.model(x_t).argmax(dim=1).cpu().numpy()

    pool = np.where(y_pred == splits.y_test)[0]
    rng = np.random.default_rng(seed)
    n = min(n_examples, len(pool))
    idx = rng.choice(pool, size=n, replace=False) if n > 0 else np.array([], dtype=int)
    if idx.size == 0:
        return pd.DataFrame()

    rows = []
    for sample_idx in idx:
        src = int(y_pred[sample_idx])
        for target in range(len(splits.class_names)):
            if target == src:
                continue
            cf = explain_counterfactual_bipolar(
                trainer._bipolar,
                x_bin,
                int(sample_idx),
                target,
                feature_names=splits.feature_names,
                class_names=splits.class_names,
                y_true=int(splits.y_test[sample_idx]),
            )
            cf['sample'] = int(sample_idx)
            cf['pred'] = splits.class_names[src]
            cf['target'] = splits.class_names[target]
            cf['valid_cf'] = int(cf['valid'])
            cf['flips'] = '; '.join(
                f"{f['feature']}: {f['from']}→{f['to']}" for f in cf.get('flips', [])
            )
            rows.append({k: cf[k] for k in ('sample', 'pred', 'target', 'valid_cf', 'n_flips', 'flips')})
    return pd.DataFrame(rows)


def build_cf_table(run: dict, **kwargs) -> pd.DataFrame:
    if run['result'].mode == 'bipolar_hyper':
        return build_cf_table_bipolar(run, **kwargs)
    return build_cf_table_instance(run, **kwargs)


def rules_dataframe(run: dict, top_n: int = 10) -> pd.DataFrame:
    df = pd.DataFrame(run['rules'][:top_n])
    if 'if' in df.columns:
        df['if'] = df['if'].apply(lambda conds: ' AND\n'.join(conds) if isinstance(conds, list) else conds)
    return df

## Par jeu : contrefactuels et règles

In [6]:
for ds_key, run in runs.items():
    mode = run['result'].mode
    display(Markdown(f"### `{ds_key}` — mode `{mode}` — contrefactuels"))
    cf_df = build_cf_table(run, n_examples=3)
    with pd.option_context('display.max_colwidth', 120, 'display.max_rows', 24):
        display(cf_df)

    display(Markdown(f"### `{ds_key}` — règles (top 10)"))
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
        display(rules_dataframe(run, top_n=10))

### `hyconex/adult` — mode `instance` — contrefactuels

,sample,pred,target,valid_cf,n_flips,flips
0,4233,0,1,1,17,"age_bin0_[0.000,0.151): 1→-1; age_bin1_[0.151,0.274): -1→1; age_bin2_[0.274,0.411): -1→1; age_bin3_[0.411,1.000): -1..."
1,592,0,1,1,19,"age_bin0_[0.000,0.151): 1→-1; age_bin1_[0.151,0.274): -1→1; age_bin2_[0.274,0.411): -1→1; age_bin3_[0.411,1.000): -1..."
2,5013,0,1,1,15,"age_bin1_[0.151,0.274): -1→1; age_bin3_[0.411,1.000): -1→1; hours_per_week_bin0_[0.000,0.398): -1→1; workclass=Gover..."


### `hyconex/adult` — règles (top 10)

,rule_id,if,then_class,score
0,6,"education=School=-1 AND\nhours_per_week_bin2_[0.449,1.000)=-1 AND\nage_bin3_[0.411,1.000)=-1 AND\nage_bin2_[0.274,0.411)=-1",0,1.000000
1,18,"hours_per_week_bin1_[0.398,0.449)=-1 AND\nworkclass=Self-Employed=+1 AND\noccupation=Professional=-1 AND\nrace=Other=-1",0,1.000000
2,22,"workclass=Government=-1 AND\nworkclass=Self-Employed=-1 AND\nage_bin3_[0.411,1.000)=+1 AND\neducation=Assoc=+1",0,0.999999
3,43,"age_bin3_[0.411,1.000)=-1 AND\nmarital_status=Separated=-1 AND\nage_bin1_[0.151,0.274)=+1 AND\nworkclass=Private=+1",0,0.999998
4,17,"occupation=Professional=-1 AND\nworkclass=Self-Employed=-1 AND\nrace=Other=-1 AND\nhours_per_week_bin2_[0.449,1.000)=+1",0,0.999966
5,11,workclass=Self-Employed=+1 AND\nmarital_status=Widowed=-1 AND\nmarital_status=Single=+1 AND\noccupation=Other/Unknown=-1,0,0.999941
6,9,"hours_per_week_bin1_[0.398,0.449)=-1 AND\noccupation=White-Collar=-1 AND\nmarital_status=Single=+1 AND\nhours_per_week_bin0_[0.000,0.398)=+1",0,0.999662
7,31,"hours_per_week_bin2_[0.449,1.000)=+1 AND\nhours_per_week_bin0_[0.000,0.398)=-1 AND\neducation=Prof-school=-1 AND\nrace=White=+1",0,0.998466
8,38,"age_bin1_[0.151,0.274)=+1 AND\nrace=White=+1 AND\ngender=Male=+1 AND\neducation=Bachelors=-1",1,0.998421
9,0,"education=Assoc=-1 AND\nhours_per_week_bin1_[0.398,0.449)=+1 AND\nmarital_status=Separated=+1 AND\nrace=Other=-1",1,0.997381


### `hyconex/audit` — mode `instance` — contrefactuels

...

,rule_id,if,then_class,score
0,34,"age_bin2_[0.676,0.824)=+1 AND\ngluc=3=+1 AND\nage_bin1_[0.529,0.676)=+1 AND\nap_hi_bin2_[0.018,1.000)=+1",1,0.999697
1,23,"height_bin1_[0.528,0.560)=-1 AND\ngender=1=-1 AND\nheight_bin3_[0.585,1.000)=+1 AND\ngender=2=+1",0,0.998256
2,21,"ap_lo_bin2_[0.014,1.000)=+1 AND\ngluc=3=+1 AND\nheight_bin2_[0.560,0.585)=+1 AND\nalco=1=+1",1,0.996279
3,32,"ap_lo_bin0_[0.000,0.014)=-1 AND\nalco=1=-1 AND\nsmoke=1=-1 AND\nap_lo_bin2_[0.014,1.000)=-1",0,0.991865
4,29,"smoke=1=+1 AND\ngluc=1=-1 AND\nage_bin2_[0.676,0.824)=-1 AND\nweight_bin0_[0.000,0.286)=-1",1,0.988931
5,25,"smoke=0=+1 AND\nactive=0=-1 AND\nheight_bin3_[0.585,1.000)=+1 AND\nweight_bin1_[0.286,0.323)=+1",1,0.987994
6,26,"height_bin3_[0.585,1.000)=-1 AND\ngluc=1=-1 AND\nweight_bin2_[0.323,0.376)=-1 AND\nalco=1=+1",0,0.986276
7,35,"age_bin3_[0.824,1.000)=-1 AND\nsmoke=0=-1 AND\nheight_bin3_[0.585,1.000)=+1 AND\ngluc=1=-1",1,0.983602
8,19,"age_bin2_[0.676,0.824)=+1 AND\ncholesterol=2=-1 AND\nalco=1=-1",1,0.982090
9,16,"weight_bin3_[0.376,1.000)=+1 AND\nap_hi_bin0_[0.000,0.017)=+1 AND\nweight_bin0_[0.000,0.286)=+1 AND\ngluc=2=+1",1,0.980500


### `hyperlogic/disease` — mode `instance` — contrefactuels

,sample,pred,target,valid_cf,n_flips,flips
0,644,39,0,0,67,symptom=acute_liver_failure: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom=back_pain: -1→1; ...
1,644,39,1,0,68,symptom=acute_liver_failure: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom=back_pain: -1→1; ...
2,644,39,2,0,65,symptom=acute_liver_failure: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom=back_pain: -1→1; ...
3,644,39,3,0,69,symptom=acute_liver_failure: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom=back_pain: -1→1; ...
4,644,39,4,0,65,symptom=acute_liver_failure: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom=bladder_discomfor...
...,...,...,...,...,...,...
115,760,3,36,0,75,symptom=acidity: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom=blackheads: -1→1; symptom=bla...
116,760,3,37,0,75,symptom=abdominal_pain: 1→-1; symptom=acidity: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom...
117,760,3,38,0,72,symptom=acidity: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom=blackheads: -1→1; symptom=bla...
118,760,3,39,0,77,symptom=abdominal_pain: 1→-1; symptom=acidity: -1→1; symptom=altered_sensorium: -1→1; symptom=anxiety: -1→1; symptom...


### `hyperlogic/disease` — règles (top 10)

,rule_id,if,then_class,score
0,68,symptom=extra_marital_contacts=-1 AND\nsymptom=dark_urine=-1 AND\nsymptom=burning_micturition=-1 AND\nsymptom=puffy_face_and_eyes=+1,26,0.309345
1,74,symptom=pain_during_bowel_movements=-1 AND\nsymptom=irritability=+1 AND\nsymptom=constipation=+1 AND\nsymptom=obesity=+1,11,0.297891
2,124,symptom=skin_rash=-1 AND\nsymptom=abnormal_menstruation=-1 AND\nsymptom=loss_of_smell=+1 AND\nsymptom=pain_during_bowel_movements=-1,35,0.282601
3,25,symptom=depression=+1 AND\nsymptom=dark_urine=-1 AND\nsymptom=loss_of_smell=+1 AND\nsymptom=irritation_in_anus=+1,33,0.281118
4,107,symptom=diarrhoea=+1 AND\nsymptom=irritability=+1 AND\nsymptom=receiving_unsterile_injections=-1 AND\nsymptom=coma=-1,26,0.268382
5,0,symptom=prominent_veins_on_calf=+1 AND\nsymptom=patches_in_throat=+1 AND\nsymptom=stomach_bleeding=-1 AND\nsymptom=skin_rash=-1,31,0.263220
6,33,symptom=indigestion=+1 AND\nsymptom=swollen_legs=+1 AND\nsymptom=coma=+1 AND\nsymptom=cold_hands_and_feets=+1,16,0.263135
7,121,symptom=diarrhoea=+1 AND\nsymptom=continuous_sneezing=-1 AND\nsymptom=muscle_pain=+1 AND\nsymptom=pus_filled_pimples=+1,28,0.260570
8,57,symptom=stiff_neck=+1 AND\nsymptom=back_pain=+1 AND\nsymptom=internal_itching=-1 AND\nsymptom=skin_rash=+1,33,0.253321
9,83,symptom=vomiting=-1 AND\nsymptom=weight_gain=-1 AND\nsymptom=history_of_alcohol_consumption=+1 AND\nsymptom=headache=-1,38,0.251302


## Comparaison rapide avec le notebook DR-HyperCF

Ce notebook utilise la **façade TabResNetDLBAC** :
- même cœur `HybridDRNetModel` en basse dimension ;
- bascule automatique vers **HyConEx bipolar + hyper TabResNet** si `n_features > 512` ;
- hyperparamètres calibrés DLBAC (température 0.7, CF warmup, early stop auto).

In [7]:
if runs:
    modes = pd.Series({k: v['result'].mode for k, v in runs.items()}).value_counts()
    print('Répartition des modes TabResNetDLBAC:')
    print(modes.to_string())

Répartition des modes TabResNetDLBAC:
instance    13
